# Clean Data Plots and Fits

This notebook explains and plots the two exported files:

- `clean_data.csv`: normal mapped packets. Each row is one accepted 34-byte uplink packet reliably assigned to one normal manual/JSON position window.
- `longer_sampling_per_position.csv`: longer-sampling packets. It contains tail samples plus the requested group 7 rows for `R1352`, `R1189`, and `R1187`.

The main measured columns are `rssi` for signal strength and `snr` for signal-to-noise ratio. The main mapping columns are `structure_id`, `manual_group`, `x_m`, and the interval timestamps used to assign a packet to a position.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, stats

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
plt.style.use("seaborn-v0_8-whitegrid")

DATA_DIR = Path("outputs/tests_gh_25_06")
clean_data = pd.read_csv(DATA_DIR / "clean_data.csv")
longer_sampling = pd.read_csv(DATA_DIR / "longer_sampling_per_position.csv")

time_cols = ["time_local", "registered_start_time", "interval_start_time", "interval_end_time"]
for frame in (clean_data, longer_sampling):
    for col in time_cols:
        if col in frame.columns:
            frame[col] = pd.to_datetime(frame[col], errors="coerce")

clean_data["mapped_row_id"] = np.arange(len(clean_data))
longer_sampling["mapped_row_id"] = np.arange(len(longer_sampling))

print(f"clean_data: {clean_data.shape[0]} rows, {clean_data.shape[1]} columns")
print(f"longer_sampling_per_position: {longer_sampling.shape[0]} rows, {longer_sampling.shape[1]} columns")

## Structure of the Files

Each row is one measured packet. The important column groups are:

- Actual mapped row: `mapped_row_id`, `source_file`, `time_local`, `fcnt`, `type`, `size`, `structure_id`, `manual_group`, `manual_group_status`, `manual_group_reliable`.
- Distance / position: `x_m` is the physical position used for plotting and fitting. It was constructed from the mapped group as `(manual_group - 5) * 15`, with group 12 shifted by `+3.5 m`. This gives a signed position. The fit below uses `abs(x_m)` because path loss depends on distance magnitude.
- Mapping interval: `registered_start_time`, `interval_start_time`, `interval_end_time`, `is_final_group`, `is_extra_tail_group` describe the time window that accepted the packet.
- Signal measurements: `rssi` is received signal strength in dBm. `snr` is signal-to-noise ratio in dB.

`longer_sampling_per_position.csv` also has `longer_sampling_source`, with `tail` for tail samples and `group_7` for the added group 7 samples.

In [ ]:
key_cols = [
    "mapped_row_id", "source_file", "time_local", "fcnt", "type", "size",
    "structure_id", "manual_group", "manual_group_status", "manual_group_reliable",
    "x_m", "registered_start_time", "interval_start_time", "interval_end_time",
    "is_final_group", "is_extra_tail_group", "rssi", "snr", "longer_sampling_source",
]

display(clean_data[[c for c in key_cols if c in clean_data.columns]].head(12))

clean_position_summary = (
    clean_data.groupby(["structure_id", "manual_group", "x_m"], dropna=False)
    .agg(rows=("mapped_row_id", "count"), mean_rssi=("rssi", "mean"), std_rssi=("rssi", "std"), mean_snr=("snr", "mean"), std_snr=("snr", "std"))
    .reset_index()
    .sort_values(["structure_id", "manual_group"])
)
display(clean_position_summary.head(20))

display(longer_sampling["longer_sampling_source"].value_counts().rename_axis("source").to_frame("rows"))
display(longer_sampling[[c for c in key_cols if c in longer_sampling.columns]].head(12))

## Physically Motivated Fit

A pure log-distance path-loss model is too short physically for this setup. The antenna is elevated on a pole, roughly in the 4-5 m range, while the tomato foliage is lower, roughly in the 3-4 m range. Close to the pole, part of the line of sight can travel above the crop. Farther into the row, the ray spends more of its path inside the foliage, so the attenuation should grow with both geometric distance and the length of crop that the signal crosses.

The model below uses a linearized dB form with non-negative attenuation constraints:

`metric = A - alpha_log log10(d_3d / d0) - alpha_foliage L_foliage`

`d_3d` is the slanted 3D path length from the antenna height to the receiver height. `L_foliage` is the approximate part of the horizontal path after the ray has entered the tomato canopy and after the crop begins along the row. For RSSI, `alpha_log` is related to the path-loss exponent, and `alpha_foliage` is an extra attenuation in dB per meter of foliage. For SNR, the same variables are used as an empirical trend because SNR also depends on noise and interference.

Yes: making the log transform first is the cleaner fit. Since RSSI is measured in dB, the usual multiplicative power-law path loss becomes linear in `log10(distance)`. Adding foliage length as another linear predictor keeps the minimization as a convex linear least-squares problem. The code uses bounded least squares so `alpha_log >= 0` and `alpha_foliage >= 0`; this avoids fits that improve numerically by making distance increase the signal, which would be physically backwards.

The height parameters are approximate. The effective crop-start distance is selected by a small grid search, and each candidate still uses the same bounded linear least-squares fit. If we measure the geometry more precisely, update the constants in the next cell. Fit quality is summarized by `R2`, adjusted `R2`, RMSE, and the fitted coefficients.

In [ ]:
ANTENNA_HEIGHT_M = 4.5
RECEIVER_HEIGHT_M = 1.5
FOLIAGE_TOP_M = 3.5
FOLIAGE_START_CANDIDATES_M = np.arange(0.0, 75.0 + 1e-9, 5.0)
REFERENCE_DISTANCE_M = 1.0


def add_foliage_geometry(df: pd.DataFrame, foliage_start_m: float = 0.0) -> pd.DataFrame:
    """Approximate how much of each path travels through tomato foliage."""
    out = df.copy()
    out["horizontal_distance_m"] = out["x_m"].abs().clip(lower=REFERENCE_DISTANCE_M)
    height_delta_m = ANTENNA_HEIGHT_M - RECEIVER_HEIGHT_M
    out["path_3d_m"] = np.sqrt(out["horizontal_distance_m"] ** 2 + height_delta_m ** 2)

    if ANTENNA_HEIGHT_M <= FOLIAGE_TOP_M or ANTENNA_HEIGHT_M == RECEIVER_HEIGHT_M:
        canopy_entry_m = foliage_start_m
    else:
        entry_fraction = (ANTENNA_HEIGHT_M - FOLIAGE_TOP_M) / (ANTENNA_HEIGHT_M - RECEIVER_HEIGHT_M)
        canopy_entry_m = out["horizontal_distance_m"] * np.clip(entry_fraction, 0.0, 1.0)

    out["foliage_start_m"] = foliage_start_m
    out["foliage_entry_m"] = np.maximum(foliage_start_m, canopy_entry_m)
    out["foliage_length_m"] = (out["horizontal_distance_m"] - out["foliage_entry_m"]).clip(lower=0.0)
    out["log10_path_3d"] = np.log10(out["path_3d_m"].clip(lower=REFERENCE_DISTANCE_M) / REFERENCE_DISTANCE_M)
    return out


def solve_bounded_log_foliage(fit_df: pd.DataFrame, metric: str) -> dict:
    y = fit_df[metric].to_numpy(dtype=float)
    X = np.column_stack([
        np.ones(len(fit_df)),
        -fit_df["log10_path_3d"].to_numpy(dtype=float),
        -fit_df["foliage_length_m"].to_numpy(dtype=float),
    ])
    result = optimize.lsq_linear(
        X,
        y,
        bounds=([-np.inf, 0.0, 0.0], [np.inf, np.inf, np.inf]),
    )
    coeffs = result.x
    y_hat = X @ coeffs
    residual = y - y_hat
    ss_res = float(np.sum(residual ** 2))
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot else np.nan
    n_rows, n_params = X.shape
    adj_r2 = 1.0 - (1.0 - r2) * (n_rows - 1) / (n_rows - n_params) if n_rows > n_params and np.isfinite(r2) else np.nan
    rmse = float(np.sqrt(np.mean(residual ** 2)))
    return {
        "metric": metric,
        "n_rows": int(n_rows),
        "intercept": float(coeffs[0]),
        "attenuation_log10_path_db_per_dec": float(coeffs[1]),
        "attenuation_foliage_db_per_m": float(coeffs[2]),
        "beta_log10_path": float(-coeffs[1]),
        "beta_foliage_db_per_m": float(-coeffs[2]),
        "path_loss_n": float(coeffs[1] / 10) if metric == "rssi" else np.nan,
        "bounded_lsq_status": int(result.status),
        "r2": float(r2),
        "adj_r2": float(adj_r2),
        "rmse": rmse,
        "antenna_height_m": ANTENNA_HEIGHT_M,
        "receiver_height_m": RECEIVER_HEIGHT_M,
        "foliage_top_m": FOLIAGE_TOP_M,
        "foliage_start_m": float(fit_df["foliage_start_m"].iloc[0]),
    }


def fit_log_foliage_model(df: pd.DataFrame, metric: str) -> dict:
    base_df = df[["x_m", metric]].dropna().copy()
    candidate_fits = []
    for foliage_start_m in FOLIAGE_START_CANDIDATES_M:
        fit_df = add_foliage_geometry(base_df, foliage_start_m=foliage_start_m)
        candidate_fits.append(solve_bounded_log_foliage(fit_df, metric))
    best_fit = min(candidate_fits, key=lambda item: item["rmse"])
    best_fit["n_foliage_start_candidates"] = len(candidate_fits)
    return best_fit


def predict_log_foliage_model(fit: dict, x_m: np.ndarray) -> np.ndarray:
    pred_df = add_foliage_geometry(pd.DataFrame({"x_m": x_m}), foliage_start_m=fit["foliage_start_m"])
    return (
        fit["intercept"]
        + fit["beta_log10_path"] * pred_df["log10_path_3d"].to_numpy()
        + fit["beta_foliage_db_per_m"] * pred_df["foliage_length_m"].to_numpy()
    )


def add_fit_panel(ax, df: pd.DataFrame, metric: str, color: str):
    fit = fit_log_foliage_model(df, metric)
    plot_df = add_foliage_geometry(df[["x_m", metric, "structure_id", "manual_group"]].dropna(), foliage_start_m=fit["foliage_start_m"]).copy()
    ax.scatter(plot_df["x_m"], plot_df[metric], s=18, alpha=0.28, color=color, label="packet rows")

    summary = plot_df.groupby("x_m", dropna=False).agg(mean=(metric, "mean"), std=(metric, "std"), n=(metric, "count")).reset_index().sort_values("x_m")
    ax.errorbar(summary["x_m"], summary["mean"], yerr=summary["std"], fmt="o", color="black", ecolor="0.65", capsize=3, label="position mean +/- std")

    x_grid = np.linspace(plot_df["x_m"].min(), plot_df["x_m"].max(), 300)
    y_grid = predict_log_foliage_model(fit, x_grid)
    ax.plot(x_grid, y_grid, color="crimson", linewidth=2.0, label="log-distance + foliage fit")

    details = f"N={fit['n_rows']}\nR2={fit['r2']:.3f}\nadj R2={fit['adj_r2']:.3f}\nRMSE={fit['rmse']:.2f} dB\nfoliage start={fit['foliage_start_m']:.0f} m\npath atten={fit['attenuation_log10_path_db_per_dec']:.2f} dB/dec\nfoliage atten={fit['attenuation_foliage_db_per_m']:.3f} dB/m"
    if metric == "rssi":
        details += f"\npath-loss n={fit['path_loss_n']:.2f}"
    ax.text(0.02, 0.03, details, transform=ax.transAxes, va="bottom", ha="left", fontsize=9, bbox=dict(facecolor="white", alpha=0.82, edgecolor="0.8"))
    ax.set_xlabel("Position x_m (m); fit uses 3D path and foliage length")
    ax.set_ylabel(metric.upper())
    ax.set_title(f"{metric.upper()} per mapped row with log-distance + foliage fit")
    ax.grid(True, axis="y", alpha=0.25)
    ax.grid(False, axis="x")
    return fit


fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
rssi_fit = add_fit_panel(axes[0], clean_data, "rssi", "tab:blue")
snr_fit = add_fit_panel(axes[1], clean_data, "snr", "tab:green")
axes[0].legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()

fit_summary = pd.DataFrame([rssi_fit, snr_fit])
display(fit_summary)

## Fit Residuals

Residuals show where the physical model misses. If residuals are randomly scattered around zero, the fit is adequate. If residuals form structure by `x_m`, `structure_id`, group, or mirror side, then even the log-distance plus foliage-length model is still missing another physical effect, such as orientation, multipath, local crop density, or gateway-side obstruction.

In [ ]:
residual_df = add_foliage_geometry(clean_data[["structure_id", "manual_group", "x_m", "rssi", "snr"]].dropna(), foliage_start_m=rssi_fit["foliage_start_m"]).copy()
for fit in (rssi_fit, snr_fit):
    metric = fit["metric"]
    residual_df[f"{metric}_fit"] = predict_log_foliage_model(fit, residual_df["x_m"].to_numpy())
    residual_df[f"{metric}_residual"] = residual_df[metric] - residual_df[f"{metric}_fit"]

residual_summary = residual_df.groupby(["structure_id", "manual_group", "x_m", "foliage_length_m"], dropna=False).agg(rows=("rssi", "count"), mean_rssi_residual=("rssi_residual", "mean"), mean_snr_residual=("snr_residual", "mean")).reset_index().sort_values(["x_m", "structure_id"])
display(residual_summary.head(30))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].scatter(residual_df["x_m"], residual_df["rssi_residual"], s=16, alpha=0.35, color="tab:blue")
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("RSSI fit residuals")
axes[0].set_xlabel("x_m (m)")
axes[0].set_ylabel("RSSI residual (dB)")
axes[1].scatter(residual_df["x_m"], residual_df["snr_residual"], s=16, alpha=0.35, color="tab:green")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("SNR fit residuals")
axes[1].set_xlabel("x_m (m)")
axes[1].set_ylabel("SNR residual (dB)")
plt.tight_layout()
plt.show()

## Longer-Sampling Distribution Study

For repeated measurements at a fixed position, RSSI and SNR often look approximately normal around a local mean because many small effects add together. Wireless measurements can also have heavier tails due to multipath, interference, timing, or orientation changes.

For each longer-sampling position, the code fits a normal distribution and a Student-t distribution. Lower AIC is better. If `delta_aic_t_minus_normal < 0`, the Student-t model is preferred; otherwise the normal model is preferred. Shapiro p-values are included as a rough normality check.

In [ ]:
def distribution_fit_table(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    rows = []
    group_cols = ["longer_sampling_source", "structure_id", "manual_group", "x_m"]
    for key, group in df.groupby(group_cols, dropna=False):
        values = group[metric].dropna().to_numpy()
        if len(values) < 3:
            continue
        source, structure_id, manual_group, x_m = key
        mu, sigma = stats.norm.fit(values)
        sigma = max(float(sigma), 1e-9)
        normal_loglik = float(np.sum(stats.norm.logpdf(values, mu, sigma)))
        normal_aic = 2 * 2 - 2 * normal_loglik
        try:
            t_df, t_loc, t_scale = stats.t.fit(values)
            t_scale = max(float(t_scale), 1e-9)
            t_loglik = float(np.sum(stats.t.logpdf(values, t_df, loc=t_loc, scale=t_scale)))
            t_aic = 2 * 3 - 2 * t_loglik
        except Exception:
            t_df, t_loc, t_scale, t_aic = np.nan, np.nan, np.nan, np.nan
        shapiro_p = stats.shapiro(values).pvalue if 3 <= len(values) <= 5000 else np.nan
        rows.append({
            "metric": metric,
            "longer_sampling_source": source,
            "structure_id": structure_id,
            "manual_group": manual_group,
            "x_m": x_m,
            "n": len(values),
            "mean": float(np.mean(values)),
            "std": float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
            "median": float(np.median(values)),
            "normal_mu": float(mu),
            "normal_sigma": float(sigma),
            "normal_aic": float(normal_aic),
            "t_df": float(t_df),
            "t_loc": float(t_loc),
            "t_scale": float(t_scale),
            "t_aic": float(t_aic),
            "delta_aic_t_minus_normal": float(t_aic - normal_aic) if np.isfinite(t_aic) else np.nan,
            "shapiro_p": float(shapiro_p),
        })
    return pd.DataFrame(rows).sort_values(["metric", "longer_sampling_source", "structure_id", "manual_group"])


distribution_summary = pd.concat([distribution_fit_table(longer_sampling, "rssi"), distribution_fit_table(longer_sampling, "snr")], ignore_index=True)
display(distribution_summary)

model_preference = (
    distribution_summary.assign(preferred=np.where(distribution_summary["delta_aic_t_minus_normal"] < 0, "student_t", "normal"))
    .groupby(["metric", "preferred"])
    .size()
    .rename("positions")
    .reset_index()
)
display(model_preference)

In [ ]:
def plot_longer_sampling_distributions(df: pd.DataFrame, metric: str):
    group_cols = ["longer_sampling_source", "structure_id", "manual_group", "x_m"]
    groups = [(key, group[metric].dropna().to_numpy()) for key, group in df.groupby(group_cols, dropna=False) if group[metric].notna().sum() >= 3]
    ncols = 3
    nrows = int(np.ceil(len(groups) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 3.4 * nrows), squeeze=False)
    axes = axes.ravel()
    for ax, (key, values) in zip(axes, groups):
        source, structure_id, manual_group, x_m = key
        ax.hist(values, bins="auto", density=True, alpha=0.62, color="tab:blue" if metric == "rssi" else "tab:green", edgecolor="white")
        mu, sigma = stats.norm.fit(values)
        sigma = max(float(sigma), 1e-9)
        x_grid = np.linspace(values.min() - 2 * sigma, values.max() + 2 * sigma, 250)
        ax.plot(x_grid, stats.norm.pdf(x_grid, mu, sigma), color="black", linewidth=1.8, label="normal fit")
        ax.axvline(np.mean(values), color="crimson", linewidth=1.2, linestyle="--", label="mean")
        ax.set_title(f"{source} {structure_id} G{int(manual_group)} x={x_m:g} m\nN={len(values)}, mean={np.mean(values):.1f}, std={np.std(values, ddof=1):.1f}")
        ax.set_xlabel(metric.upper())
        ax.set_ylabel("Density")
        ax.grid(True, axis="y", alpha=0.25)
        ax.grid(False, axis="x")
    for ax in axes[len(groups):]:
        ax.axis("off")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right")
    fig.suptitle(f"Longer-sampling {metric.upper()} distributions by mapped row/position", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()


plot_longer_sampling_distributions(longer_sampling, "rssi")
plot_longer_sampling_distributions(longer_sampling, "snr")

## Distribution Interpretation

Use the `distribution_summary` table together with the histograms:

- A roughly symmetric histogram with lower normal AIC means a normal model is reasonable.
- A lower Student-t AIC means heavier tails are likely, so occasional strong or weak packets occur more often than Gaussian noise would predict.
- If `N` is small, treat model choice cautiously. The longer-sampling rows help because larger `N` makes this distribution study more meaningful.

A small standard deviation and normal-looking histogram indicate stable reception at that position. Large spread or heavy tails suggest transient propagation, multipath, interference, or packet-level variability.